### [DENEY 4]
**Amaç:** Deney 3'teki başarının sadece epoch artışından mı yoksa mimariden mi kaynaklandığını doğrulamak için yapılan kontrol deneyi.
**Bu Hücrede Değişen Parametreler / Yapılan Ayarlar:**
- **ELA Ölçek Faktörü (`scale`):** 15
- **Veri Kümesi Bölütleme (Split):** %80 Eğitim, %16 Test, %4 Validasyon
- **Model Yapısı:** Sadece önerilen geliştirilmiş **M2 mimarisi** çalıştırıldı.
- **Epoch Sayısı:** 40 (Deney 3'teki 60 epoch'luk eğitime kıyasla mimarinin 40 epoch'taki kararlılığı ölçüldü; %94.47 doğruluk alındı)

In [ ]:
import os
import numpy as np
import random
from PIL import Image, ImageChops, ImageEnhance
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import classification_report

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0c19a9db6740e7da1d8da96086f765c2'
!pip install kaggle -q
!kaggle datasets download -d sophatvathana/casia-dataset -q
!unzip -q -o casia-dataset.zip -d casia
print("Veri hazır!")

def image_to_ela(path, quality=90):
    try:
        image = Image.open(path).convert('RGB')
        image.save('resaved.jpg', 'JPEG', quality=quality)
        resaved = Image.open('resaved.jpg')
        ela_image = ImageChops.difference(image, resaved)
        band_values = ela_image.getextrema()
        max_value = max([val[1] for val in band_values])
        if max_value == 0:
            max_value = 1
        scale = 255.0 / max_value * 15  # scale=15
        ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
        return ela_image
    except Exception as e:
        print(f'Hata: {e}')
        return None

image_size = (150, 150)

def prepare_image(path):
    ela = image_to_ela(path)
    if ela is None:
        return np.zeros((150*150*3,))
    return np.array(ela.resize(image_size)).flatten() / 255.0

X, Y = [], []
path = 'casia/CASIA2/Au'
for filename in os.listdir(path):
    if filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg'):
        X.append(prepare_image(os.path.join(path, filename)))
        Y.append(1)
        if len(Y) % 1000 == 0:
            print(f'{len(Y)} görüntü işlendi')
print(f"Orijinal: {len(X)}")

path = 'casia/CASIA2/Tp'
for filename in os.listdir(path):
    if filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg'):
        X.append(prepare_image(os.path.join(path, filename)))
        Y.append(0)
        if len(Y) % 1000 == 0:
            print(f'{len(Y)} görüntü işlendi')
print(f"Toplam: {len(X)}")

for i in range(10):
    X, Y = shuffle(X, Y, random_state=i)

X = np.array(X).reshape(-1, 150, 150, 3)
Y = to_categorical(Y, 2)
print(f"X shape: {X.shape}, Y shape: {Y.shape}")

X_train, X_rest, Y_train, Y_rest = train_test_split(X, Y, test_size=0.2, random_state=42)
del X, Y
X_test, X_val, Y_test, Y_val = train_test_split(X_rest, Y_rest, test_size=0.2, random_state=42)
del X_rest, Y_rest
print(f"Train: {len(X_train)}, Test: {len(X_test)}, Val: {len(X_val)}")

model2 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Conv2D(64, (5,5), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (5,5), padding='same', activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='sigmoid')
])

model2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(), Recall()]
)
model2.summary()

history2 = model2.fit(
    X_train, Y_train,
    epochs=40,          # 60 → 40
    batch_size=8,
    validation_data=(X_val, Y_val),
    verbose=1
)
print("Model 2 tamamlandı!")

loss2, acc2, prec2, rec2 = model2.evaluate(X_test, Y_test)
print(f"\n=== Model 2 - scale=15, epoch=40 ===")
print(f"Test Accuracy: {acc2:.4f}")
print(f"Precision:     {prec2:.4f}")
print(f"Recall:        {rec2:.4f}")

y_pred2 = model2.predict(X_test)
y_pred2_labels = np.argmax(y_pred2, axis=1)
y_true_labels = np.argmax(Y_test, axis=1)
print(classification_report(y_true_labels, y_pred2_labels,
      target_names=['Sahte', 'Orijinal']))

print("=" * 50)
print("KARŞILAŞTIRMA")
print("=" * 50)
print(f"Makale              : %94.14")
print(f"Model 2 (epoch=60)  : %95.53")
print(f"Model 2 (epoch=40)  : %{acc2*100:.2f}")
print("=" * 50)